# Is conformal coverage *free*? — `FreeCoverageDiagnostic`

Before spending effort **reducing** the Price of Coverage (e.g. with `DICA`), ask the prior
question: **does conformal coverage change your decision at all?**

If it does not, the calibrated uncertainty set is a *free certificate* — 90% coverage tracking
at **zero cost** to the decision. `FreeCoverageDiagnostic` runs a short calibration pilot with
your predictor and your LP and returns a **free / critical / costly** verdict, plus the measured
decision-neutral fraction at two granularities:

- **committed-set** neutrality — does the *set of active variables* (support) match?
- **vector** neutrality — is the *full* decision vector identical?

These differ: a decision can be committed-set-stable at 100% while a fractional vertex still shifts.

*Method: "When Is Conformal Coverage Free? Switching Thresholds for Predict-then-Optimize" (COPA 2026).*

In [1]:
import numpy as np
from conformal_ops import FreeCoverageDiagnostic

def selection_lp(d):
    # min c^T x  s.t.  sum(x)=1, 0<=x<=1   (optimum is a vertex e_k)
    return dict(A_eq=np.ones((1, d)), b_eq=np.array([1.0]), bounds=[(0.0, 1.0)] * d)

def make_stream(T, base, true_noise, pred_noise, seed):
    # pred_noise may be scalar or a per-component (d,) vector
    rng = np.random.RandomState(seed)
    d = len(base); pred_noise = np.broadcast_to(pred_noise, (d,))
    Cp = np.zeros((T, d)); Ct = np.zeros((T, d))
    for t in range(T):
        ct = base + rng.normal(0, true_noise, d)
        Ct[t] = ct; Cp[t] = ct + rng.normal(0, 1.0, d) * pred_noise
    return Cp, Ct

d = 5

## Case 1 — wide cost gap + accurate predictor → **free**

Item 0 is clearly cheapest (gap 0.9) and the predictor is accurate, so the small conformal radii
never reorder the argmin.

In [2]:
Cp, Ct = make_stream(200, np.array([0.1, 1., 1., 1., 1.]), 0.01, 0.02, seed=0)
print(FreeCoverageDiagnostic(alpha=0.10).run(Cp, Ct, **selection_lp(d)))

FreeCoverageReport(regime='free', basis=empirical)
  decision-neutral (committed set): 100.0%
  decision-neutral (full vector):   100.0%
  coverage:                          90.0%
  q* = 3.5163   sigma_q = 0.9771
  rounds = 180 (warmup 20)


## Case 2 — near-tied costs + **heterogeneous** noise → **costly**

Item 1 is often the nominal cheapest but by far the noisiest, so its conformal radius is large and
the robust optimum keeps switching away. (Homogeneous noise would *not* do this — uniform radii
preserve the argmin. The heterogeneity is the whole point.)

In [3]:
Cp, Ct = make_stream(200, np.array([0.50, 0.48, 0.51, 0.50, 0.505]), 0.01,
                     np.array([0.03, 0.60, 0.03, 0.03, 0.03]), seed=1)
print(FreeCoverageDiagnostic(alpha=0.10).run(Cp, Ct, **selection_lp(d)))

FreeCoverageReport(regime='costly', basis=empirical)
  decision-neutral (committed set):  46.7%
  decision-neutral (full vector):    46.7%
  coverage:                          91.0%
  q* = 3.0630   sigma_q = 0.4928
  rounds = 180 (warmup 20)


## Case 3 — analytic switching threshold κ* and safety margin

Supply a `competitors` oracle (the competing vertices against which κ* is measured — here the other
unit vectors) to get an analytic κ* and margin `m = (κ* − q*) / σ_q`. With equal noise scale across
competitors, K⁺ is empty, κ* = ∞, and coverage is free by construction.

In [4]:
Cp, Ct = make_stream(200, np.array([0.1, 1., 1., 1., 1.]), 0.01, 0.02, seed=0)

def competitors(c_rep, x_nom):
    star = int(np.argmax(x_nom))
    return [np.eye(d)[k] for k in range(d) if k != star]

print(FreeCoverageDiagnostic(alpha=0.10).run(Cp, Ct, competitors=competitors, **selection_lp(d)))

FreeCoverageReport(regime='free', basis=margin)
  decision-neutral (committed set): 100.0%
  decision-neutral (full vector):   100.0%
  coverage:                          90.0%
  q* = 3.5163   sigma_q = 0.9771
  kappa* = inf   margin m = inf
  rounds = 180 (warmup 20)


## Takeaway

Run this **before** deploying conformal hedging on a predict-then-optimize pipeline:

- verdict **free** → coverage is a zero-cost certificate; deploy the monitoring with confidence;
- verdict **costly** → budget the premium, or **reduce** it with `DICA`;
- verdict **critical** → borderline; monitor coverage closely.

`FreeCoverageDiagnostic` tells you *whether* you have a Price of Coverage; `DICA` *reduces* it.